# Comparison figures
Run after `arm_comparison.ipynb`. Every input is discovered on Drive, and a figure is skipped with a printed reason when its input is absent rather than drawn from partial data.

Labels are English because Colab has no Korean matplotlib font by default.

Figures: training curves, the dropout/delay trade-off, paired per-seed differences, and how the gap moves across checkpoints.

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/CNN-RL-improved")
ARM_DIRS = {"raw_direct": ROOT / "raw-direct-830k-seed0" / "ppo",
            "candidate_cnn": ROOT / "scale-aware-cnn-6h-seed0" / "ppo"}
COMPARISON_DIR = ROOT / "comparison-830k"
FIGURE_DIR = COMPARISON_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

inputs = {}
for arm, directory in ARM_DIRS.items():
    for name in ("training_log.csv", "holdout_selection.csv", "loss_log.csv"):
        path = directory / name
        if path.is_file():
            inputs[(arm, name)] = path
summaries = sorted(COMPARISON_DIR.glob("arm_comparison_summary_*.json"))
rows_files = sorted(COMPARISON_DIR.glob("arm_comparison_rows_*.csv"))

for arm in ARM_DIRS:
    found = [name for (owner, name) in inputs if owner == arm]
    print(f"{arm:<14} {found or 'NOTHING FOUND - check the path above'}")
print("comparison summaries:", [p.name for p in summaries] or "none yet")
print("comparison rows:", [p.name for p in rows_files] or "none yet")


In [ ]:
import json
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.ticker import FuncFormatter

# Validated categorical pair; one hue per arm, held constant across every figure.
ARM_COLOR = {"raw_direct": "#eb6834", "candidate_cnn": "#2a78d6"}
ARM_LABEL = {"raw_direct": "raw-direct (no extractor)", "candidate_cnn": "candidate CNN + MLP"}
INK, MUTED, GRID = "#0b0b0b", "#898781", "#e1e0d9"
# Raw timestep labels collide at this width.
STEP_TICKS = FuncFormatter(lambda value, _: "0" if value == 0 else f"{value / 1000:.0f}k")
matplotlib.rcParams.update({
    "figure.dpi": 130, "savefig.dpi": 130, "figure.facecolor": "#fcfcfb",
    "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "axes.edgecolor": GRID, "axes.labelcolor": MUTED, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.grid": True,
    "grid.color": GRID, "grid.linewidth": 0.8, "axes.spines.top": False,
    "axes.spines.right": False, "font.size": 9, "axes.titlesize": 11,
    "axes.titleweight": "600", "legend.frameon": False,
})

training = {arm: pd.read_csv(inputs[(arm, "training_log.csv")])
            for arm in ARM_DIRS if (arm, "training_log.csv") in inputs}
holdout = {arm: pd.read_csv(inputs[(arm, "holdout_selection.csv")])
           for arm in ARM_DIRS if (arm, "holdout_selection.csv") in inputs}
comparisons = {}
for path in summaries:
    payload = json.loads(path.read_text(encoding="utf-8"))
    comparisons[int(payload["common_timestep"])] = payload
print("loaded:", {arm: len(frame) for arm, frame in training.items()},
      "| holdout points:", {arm: len(frame) for arm, frame in holdout.items()},
      "| comparison timesteps:", sorted(comparisons))


In [ ]:
# Figure 1: training curves, with each arm's holdout evaluations on the same axis.
if len(training) < 2:
    print("skipped: training_log.csv missing for", [a for a in ARM_DIRS if a not in training])
else:
    WINDOW = 30
    figure, axis = plt.subplots(figsize=(9, 4.4))
    for arm, frame in training.items():
        frame = frame.sort_values("timestep")
        smooth = frame["terminal_score"].rolling(WINDOW, min_periods=WINDOW).mean()
        axis.scatter(frame["timestep"], frame["terminal_score"], s=3,
                     color=ARM_COLOR[arm], alpha=0.16, linewidths=0)
        axis.plot(frame["timestep"], smooth, color=ARM_COLOR[arm], linewidth=2,
                  label=f"{ARM_LABEL[arm]} (training, {WINDOW}-episode mean)")
        final = smooth.dropna()
        if len(final):
            axis.annotate(f"{final.iloc[-1]:+.2f}",
                          (frame["timestep"].iloc[-1], final.iloc[-1]),
                          textcoords="offset points", xytext=(6, 0), fontsize=9,
                          fontweight="600", color=ARM_COLOR[arm])
    for arm, frame in holdout.items():
        axis.plot(frame["timestep"], frame["mean_terminal_score"], marker="o",
                  markersize=4, linewidth=1, linestyle="--", alpha=0.75,
                  color=ARM_COLOR[arm], label=f"{ARM_LABEL[arm]} (holdout, 5 scenarios)")
    axis.axhline(0, color="#c3c2b7", linewidth=1, linestyle=":")
    axis.set(xlabel="timestep", ylabel="terminal score",
             title="Training and holdout terminal score, both arms")
    axis.xaxis.set_major_formatter(STEP_TICKS)
    axis.legend(loc="lower right", fontsize=8)
    figure.tight_layout()
    figure.savefig(FIGURE_DIR / "comparison_training_curves.png")
    plt.show()


In [ ]:
# Figure 2: the trade the score decomposition exposed - dropouts cost -2.0 each,
# delays at most -1.0, so shorter delays cannot pay for extra dropouts.
if len(training) < 2:
    print("skipped: needs both training logs")
else:
    BIN = 100_000
    figure, axes = plt.subplots(1, 2, figsize=(9.6, 3.8), sharex=True)
    for column, axis, title in (("dropout_count", axes[0], "Dropouts per episode (-2.0 each)"),
                                ("delayed_count", axes[1], "Delayed blocks per episode")):
        for arm, frame in training.items():
            binned = (frame.assign(bin=(frame["timestep"] // BIN) * BIN)
                      .groupby("bin")[column].mean())
            axis.plot(binned.index + BIN / 2, binned.values, marker="o", markersize=4,
                      linewidth=2, color=ARM_COLOR[arm], label=ARM_LABEL[arm])
            axis.annotate(f"{binned.iloc[-1]:.0f}", (binned.index[-1] + BIN / 2, binned.iloc[-1]),
                          textcoords="offset points", xytext=(6, 0), fontsize=9,
                          fontweight="600", color=ARM_COLOR[arm])
        axis.set(xlabel="timestep", title=title, ylim=(0, None))
        axis.xaxis.set_major_formatter(STEP_TICKS)
    axes[0].set_ylabel("blocks per 913-block episode")
    axes[0].legend(loc="upper right", fontsize=8)
    figure.tight_layout()
    figure.savefig(FIGURE_DIR / "comparison_failure_modes.png")
    plt.show()


In [ ]:
# Figure 3: paired per-seed differences at the largest compared checkpoint.
if not comparisons:
    print("skipped: no arm_comparison_summary_*.json yet")
else:
    timestep = max(comparisons)
    headline = comparisons[timestep]["partitions"]["primary_test"]
    seeds = headline["seeds"]
    figure, axes = plt.subplots(1, 3, figsize=(11.5, 4.3))
    for axis, metric in zip(axes, ("mean_terminal_score", "mean_dropout_rate", "mean_delay_days")):
        values = headline["paired"][metric]
        differences = values["per_seed_difference"]
        order = sorted(range(len(differences)), key=lambda index: differences[index])
        low, high = values["bootstrap_ci_95"]
        better = "lower" if values["lower_is_better"] else "higher"
        for position, index in enumerate(order):
            difference = differences[index]
            favours = (difference < 0) if values["lower_is_better"] else (difference > 0)
            color = ARM_COLOR["candidate_cnn"] if favours else ARM_COLOR["raw_direct"]
            axis.plot([0, difference], [position, position], color=color, linewidth=1.4, alpha=0.55)
            axis.plot(difference, position, marker="o", markersize=4, color=color)
        axis.axvline(0, color="#c3c2b7", linewidth=1)
        axis.axvspan(low, high, color=ARM_COLOR["candidate_cnn"], alpha=0.10)
        axis.axvline(values["mean_difference"], color=INK, linewidth=1.4, linestyle="--")
        axis.set_yticks(range(len(order)))
        axis.set_yticklabels([f"seed {seeds[index]}" for index in order], fontsize=7)
        axis.set(title=f"{metric.replace('mean_', '')}\n({better} is better for CNN)",
                 xlabel=f"candidate CNN minus raw-direct\n"
                        f"mean {values['mean_difference']:+.3f}   "
                        f"CI [{low:+.3f}, {high:+.3f}]   "
                        f"CNN better {values['seeds_favouring_candidate']}/{values['seed_count']}")
        axis.xaxis.label.set_fontsize(8)
    figure.suptitle(f"Paired differences on the 15 primary-test seeds at timestep {timestep:,}",
                    fontsize=11, fontweight="600")
    figure.tight_layout()
    figure.savefig(FIGURE_DIR / "comparison_paired_seeds.png")
    plt.show()


In [ ]:
# Figure 4: does the gap hold across training, or appear only late?
if len(comparisons) < 2:
    print("skipped: needs at least two compared checkpoints; found", sorted(comparisons))
else:
    figure, axis = plt.subplots(figsize=(7.5, 3.8))
    steps = sorted(comparisons)
    values = [comparisons[step]["partitions"]["primary_test"]["paired"]["mean_terminal_score"]
              for step in steps]
    means = [value["mean_difference"] for value in values]
    lower = [mean - value["bootstrap_ci_95"][0] for mean, value in zip(means, values)]
    upper = [value["bootstrap_ci_95"][1] - mean for mean, value in zip(means, values)]
    axis.errorbar(steps, means, yerr=[lower, upper], marker="o", markersize=5,
                  linewidth=2, capsize=4, color=ARM_COLOR["candidate_cnn"])
    axis.axhline(0, color="#c3c2b7", linewidth=1)
    for step, mean, value in zip(steps, means, values):
        axis.annotate(f"{value['seeds_favouring_candidate']}/{value['seed_count']}",
                      (step, mean), textcoords="offset points", xytext=(0, 10),
                      ha="center", fontsize=8, color=MUTED)
    axis.set(xlabel="timestep", ylabel="terminal score difference",
             title="Candidate CNN minus raw-direct, 95% bootstrap interval\n"
                   "(labels: seeds favouring the CNN)")
    axis.xaxis.set_major_formatter(STEP_TICKS)
    figure.tight_layout()
    figure.savefig(FIGURE_DIR / "comparison_trend.png")
    plt.show()
print("figures saved to", FIGURE_DIR)
print(sorted(path.name for path in FIGURE_DIR.glob("*.png")))
